# Final Project!

Author: Cameron Mullaney

In [25]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from Proj2Script import *

## Pyspark MLlib-specific tools for model evaluation
from pyspark.ml.regression import GBTRegressor, LinearRegression, RandomForestRegressor
from pyspark.ml.linalg import Vectors
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, VectorIndexer, OneHotEncoder, PCA
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator 

## Fitting the Model

Starting the spark session

In [8]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

Reading the data in as a pandas DF, then converting to spark DF

In [33]:
power = pd.read_csv("power_ml_data.csv")
power = spark.createDataFrame(power) 
power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Looks good! Let's get to transforming

### Builing the Pipeline

For `hour`, I am casting it to a double, and then binarizing it with a cutoff of 6.5

In [19]:
hour_double = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

In [20]:
hour_bin = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "Hour_b")

For `month`, I am casting it to a double so I can One-hot encode it

In [67]:
month_double = SQLTransformer(
    statement="SELECT *, CAST(Month AS DOUBLE) AS Month_d FROM __THIS__"
)

In [68]:
month_ohe = OneHotEncoder(inputCol = "Month_d", outputCol = "Month_ohe")

Here I'm creating my `label` column

In [45]:
label_cast = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

Here I'm creating the list of variables I will send to the PCA

In [30]:
PCA_assembler = VectorAssembler(inputCols = ["Temperature", 
                                         "Humidity", 
                                         "Wind_Speed", 
                                         "General_Diffuse_Flows",
                                         "Diffuse_Flows"], 
                            outputCol = "pca_input", 
                            handleInvalid = 'keep')

In [44]:
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features")

In [61]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



And here I'm running the PCA_assembler to fit the PCA.

In [34]:
temp = PCA_assembler.transform(
    sqlTrans.transform(power))

pca_fit = pca.fit(temp)        

This is the final assembler, creating the final "features" column for later fitting.

In [55]:
assembler = VectorAssembler(inputCols = ["pca_features", 
                                         "Hour_b", 
                                         "Power_Zone_1", 
                                         "Power_Zone_2",
                                         "Month_ohe"], 
                            outputCol = "features", 
                            handleInvalid = 'keep')

Time to test-run the transformations:

In [72]:
df = hour_double.transform(power)
df = hour_bin.transform(df)
df = month_double.transform(df)
df = month_ohe.fit(df).transform(df)
df = PCA_assembler.transform(df)
df = pca_fit.transform(df)
df = label_cast.transform(df)
temp = assembler.transform(df)

In [78]:
temp.show(5, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_d|Hour_b|Month_d|Month_ohe     |pca_input                     |pca_features                            |label      |features                                                                             |
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|6.559      |73.8  

This looks good! Time to create our Pipeline

First I'll define what kind of model we want

In [82]:
lr = LinearRegression(featuresCol = "features", labelCol = "label")

In [83]:
pipe_1 = Pipeline(stages = [hour_double, hour_bin, month_double, month_ohe, PCA_assembler, pca, label_cast, assembler, lr])

### Building and fitting the model

Creating our list of potential parameters

In [84]:
LR_paramGrid = ParamGridBuilder()\
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .build()


In [88]:
LRCV = CrossValidator(estimator = pipe_1,
                      estimatorParamMaps = LR_paramGrid,
                      evaluator = RegressionEvaluator(metricName = "rmse"),
                      numFolds=5,
                      parallelism = 2,
                      seed = 12)

This is going to take a while - and produce many warnings.

In [ ]:
LRCV_model = LRCV.fit(power)

### Reporting model values

Okay! Let's see what our optimal values are

In [100]:
LR_list = []
for i in range(len(LR_paramGrid)):
    LR_list.append([LRCV_model.avgMetrics[i], LR_paramGrid[i].values()])
LR_list.sort(key=lambda x: x[0])
print("CV Error \t\t\t [regParam, ElasticParam]")
LR_list[0]

CV Error 			 [regParam, ElasticParam]


[2147.8799826780473, dict_values([0.1, 0.1])]

In [106]:
LR_pred = LRCV_model.transform(power)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(LR_pred)
print("Training RMSE: \t", training_rmse)

Training RMSE: 	 2147.097320321586


In [111]:
resid_df = LR_pred.withColumn("residual", col("label") - col("prediction"))
resid_df.select("residual", "label", "prediction").show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
|-638.6844049043611|20240.96386| 20879.64826490436|
|1471.1367385458434|20131.08434|18659.947601454158|
|1463.9585040228303|19668.43373| 18204.47522597717|
| 1308.869631235666|18899.27711|17590.407478764333|
|1445.3432101498693|18442.40964|16997.066429850132|
|1612.6517153935165|18130.12048|16517.468764606485|
|1852.0122508874556|17945.06024|16093.047989112543|
|1736.7707088121715|17459.27711|15722.506401187828|
| 1754.669697622121|17025.54217| 15270.87247237788|
| 1856.033343777257|16794.21687|14938.183526222743|
|1985.7932322314336|16638.07229|14652.279057768566|
|1980.3767026612622|16395.18072|14414.804017338738|
|2034.8443820288285|16117.59036|14082.745977971172|
|2197.8897606054707| 15822.6506| 13624.76083939453|
|2222.0367874454423|15672.28916|13450.252372554558|
| 2294.906820974167|15597.10843|13302.201609025833|
| 2355.57556

In [108]:
LR_pred.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+--------------------+--------------------+-----------+--------------------+------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_d|Hour_b|Month_d|     Month_ohe|           pca_input|        pca_features|      label|            features|        prediction|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+--------------------+--------------------+-----------+--------------------+------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|   0.0|   0.0|    1.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|20240.96386|(17,[0,1,3,4,6],[...| 20879.648264904